# 12 - Predict Single Image
Browse ảnh từ ổ đĩa → YOLO detect → Crop → MobileNet predict

In [1]:
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
from ultralytics import YOLO

In [6]:
# ── Config ────────────────────────────────────────────────────────────
YOLO_PATH      = "../models/yolov8n_rice_leaf.pt"
MOBILENET_PATH = "../models/rice_disease_classifier.keras"
CONF_THRESHOLD = 0.5
PADDING        = 10

CLASS_NAMES = [
    "bacterial_blight",
    "barrow_brown_leaf_spot",
    "brown_spot",
    "healthy",
    "leaf_blast",
    "leaf_scald",
    "leaf_smut",
    # "neck_blast",
    "rice_hispa",
    "sheath_blight",
    "tungro",
]

CLASS_INFO = {
    "bacterial_blight"      : {"vi": "Bạc lá vi khuẩn",    "severity": "Cao",      "color": "#e74c3c"},
    "barrow_brown_leaf_spot": {"vi": "Đốm nâu Barrow",     "severity": "Trung bình","color": "#e67e22"},
    "brown_spot"            : {"vi": "Đốm nâu",             "severity": "Trung bình","color": "#e67e22"},
    "healthy"               : {"vi": "Lá khỏe mạnh",        "severity": "Không",    "color": "#27ae60"},
    "leaf_blast"            : {"vi": "Đạo ôn lá",           "severity": "Cao",      "color": "#e74c3c"},
    "leaf_scald"            : {"vi": "Cháy bìa lá",         "severity": "Trung bình","color": "#e67e22"},
    "leaf_smut"             : {"vi": "Nấm lá",              "severity": "Thấp",     "color": "#f1c40f"},
    # "neck_blast"            : {"vi": "Đạo ôn cổ bông",      "severity": "Rất cao",  "color": "#c0392b"},
    "rice_hispa"            : {"vi": "Bọ trĩ lúa (Hispa)", "severity": "Trung bình","color": "#e67e22"},
    "sheath_blight"         : {"vi": "Khô vằn",             "severity": "Cao",      "color": "#e74c3c"},
    "tungro"                : {"vi": "Vàng lùn Tungro",     "severity": "Rất cao",  "color": "#c0392b"},
}

In [7]:
# ── Load models (chạy 1 lần) ──────────────────────────────────────────
yolo      = YOLO(YOLO_PATH)
mobilenet = tf.keras.models.load_model(MOBILENET_PATH, compile=False)
print(" YOLO     :", YOLO_PATH)
print(" MobileNet:", MOBILENET_PATH)

ValueError: Layer "dense" expects 1 input(s), but it received 2 input tensors. Inputs received: [<KerasTensor shape=(None, 7, 7, 1280), dtype=float32, sparse=False, ragged=False, name=keras_tensor_1290>, <KerasTensor shape=(None, 7, 7, 1280), dtype=float32, sparse=False, ragged=False, name=keras_tensor_1291>]

In [ ]:
# ── Browse file dialog + predict + visualize ──────────────────────────
# Chạy cell này mỗi khi muốn test 1 ảnh mới

import tkinter as tk
from tkinter import filedialog

# ── Mở cửa sổ chọn file ───────────────────────────────────────────────
root = tk.Tk()
root.withdraw()          # Ẩn cửa sổ chính
root.attributes('-topmost', True)  # Hiện dialog trên cùng

IMAGE_PATH = filedialog.askopenfilename(
    title="Chọn ảnh lá lúa",
    filetypes=[
        ("Image files", "*.jpg *.jpeg *.png *.bmp *.tiff *.webp"),
        ("All files", "*.*")
    ]
)
root.destroy()

if not IMAGE_PATH:
    print("❌ Không có ảnh nào được chọn.")
else:
    print(f" Ảnh đã chọn: {IMAGE_PATH}")

    # ── Step 1: Load ảnh ──────────────────────────────────────────────
    img = cv2.imread(IMAGE_PATH)
    assert img is not None, f"Không đọc được ảnh: {IMAGE_PATH}"
    h_orig, w_orig = img.shape[:2]
    t_start = time.perf_counter()

    # ── Step 2: YOLO detect ───────────────────────────────────────────
    t0          = time.perf_counter()
    yolo_res    = yolo.predict(img, conf=CONF_THRESHOLD, verbose=False)
    t_yolo      = (time.perf_counter() - t0) * 1000
    boxes       = yolo_res[0].boxes
    box_xyxy    = None
    yolo_conf   = 0.0
    yolo_detected = False

    if boxes is not None and len(boxes) > 0:
        xyxy  = boxes.xyxy.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        areas = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
        best  = np.argmax(areas)
        x1, y1, x2, y2 = xyxy[best].astype(int)
        x1 = max(0, x1 - PADDING);      y1 = max(0, y1 - PADDING)
        x2 = min(w_orig, x2 + PADDING); y2 = min(h_orig, y2 + PADDING)
        box_xyxy      = (x1, y1, x2, y2)
        yolo_conf     = float(confs[best])
        yolo_detected = True
        crop = img[y1:y2, x1:x2]
    else:
        crop = img.copy()   # fallback: dùng nguyên ảnh

    # ── Step 3: Resize crop → 224×224 ────────────────────────────────
    crop_224 = cv2.resize(crop, (224, 224))

    # ── Step 4: MobileNet classify ────────────────────────────────────
    t1      = time.perf_counter()
    img_rgb = cv2.cvtColor(crop_224, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    probs   = mobilenet.predict(np.expand_dims(img_rgb, 0), verbose=0)[0]
    t_cls   = (time.perf_counter() - t1) * 1000
    t_total = (time.perf_counter() - t_start) * 1000

    pred_idx   = int(np.argmax(probs))
    pred_class = CLASS_NAMES[pred_idx]
    pred_conf  = float(probs[pred_idx])
    info       = CLASS_INFO[pred_class]

    # ── In kết quả ────────────────────────────────────────────────────
    print("\n" + "═"*55)
    print("  KẾT QUẢ DỰ ĐOÁN")
    print("═"*55)
    print(f"  Bệnh (EN)      : {pred_class}")
    print(f"  Bệnh (VI)      : {info['vi']}")
    print(f"  Độ tin cậy     : {pred_conf*100:.1f}%")
    print(f"  Mức độ nguy hiểm: {info['severity']}")
    print(f"  YOLO detect    : {'Có (' + str(round(yolo_conf*100)) + '%)' if yolo_detected else 'Không (dùng ảnh gốc)'}")
    print(f"  Latency YOLO   : {t_yolo:.0f} ms")
    print(f"  Latency MNet   : {t_cls:.0f} ms")
    print(f"  Tổng latency   : {t_total:.0f} ms")
    print("═"*55)

    # ── Visualize ─────────────────────────────────────────────────────
    fig = plt.figure(figsize=(16, 5))

    # [A] Ảnh gốc + bbox
    ax1 = fig.add_subplot(1, 3, 1)
    img_show = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax1.imshow(img_show)
    if box_xyxy:
        x1, y1, x2, y2 = box_xyxy
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=3, edgecolor='#00ff88', facecolor='none'
        )
        ax1.add_patch(rect)
        ax1.text(
            x1, max(y1-8, 0),
            f"rice_leaf  {yolo_conf*100:.0f}%",
            color='white', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', fc='#00aa55', ec='none', alpha=0.85)
        )
    ax1.set_title(
        f"① Ảnh gốc\nYOLO: {'detected' if yolo_detected else 'fallback (full image)'}",
        fontsize=10
    )
    ax1.axis('off')

    # [B] Crop 224×224
    ax2 = fig.add_subplot(1, 3, 2)
    ax2.imshow(cv2.cvtColor(crop_224, cv2.COLOR_BGR2RGB))
    ax2.set_title('② Crop 224×224\n→ input MobileNet', fontsize=10)
    ax2.axis('off')

    # [C] Top-5 probability bar
    ax3 = fig.add_subplot(1, 3, 3)
    top5_idx   = np.argsort(probs)[::-1][:5]
    top5_probs = probs[top5_idx]
    top5_names = [CLASS_NAMES[i] for i in top5_idx]
    bar_colors = [info['color'] if n == pred_class else '#95a5a6' for n in top5_names]

    bars = ax3.barh(top5_names[::-1], top5_probs[::-1] * 100, color=bar_colors[::-1])
    ax3.set_xlabel('Probability (%)', fontsize=9)
    ax3.set_title('③ Top-5 Predictions', fontsize=10)
    ax3.set_xlim(0, 105)
    for bar, prob in zip(bars, top5_probs[::-1]):
        ax3.text(
            prob*100 + 0.5, bar.get_y() + bar.get_height()/2,
            f'{prob*100:.1f}%', va='center', fontsize=8
        )

    plt.suptitle(
        f"{pred_class}  ({pred_conf*100:.1f}%)\n{info['vi']}  —  Mức độ: {info['severity']}",
        fontsize=13, fontweight='bold', color=info['color']
    )
    plt.tight_layout()
    plt.savefig('prediction_result.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(" Đã lưu → prediction_result.png")

In [ ]:
# ── Browse file dialog + predict + visualize ──────────────────────────
# Chạy cell này mỗi khi muốn test 1 ảnh mới

import tkinter as tk
from tkinter import filedialog

# ── Mở cửa sổ chọn file ───────────────────────────────────────────────
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

IMAGE_PATH = filedialog.askopenfilename(
    title="Chọn ảnh lá lúa",
    filetypes=[
        ("Image files", "*.jpg *.jpeg *.png *.bmp *.tiff *.webp"),
        ("All files", "*.*")
    ]
)
root.destroy()

if not IMAGE_PATH:
    print("❌ Không có ảnh nào được chọn.")
else:
    print(f" Ảnh đã chọn: {IMAGE_PATH}")

     # ── Hiển thị ảnh đã chọn ──────────────────────────────────────────
    img_preview = cv2.imread(IMAGE_PATH)
    plt.figure(figsize=(5, 5))
    plt.imshow(cv2.cvtColor(img_preview, cv2.COLOR_BGR2RGB))
    plt.title(f"Ảnh đã chọn\n{IMAGE_PATH.split('/')[-1]}", fontsize=10)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # ── Step 1: Load ảnh ──────────────────────────────────────────────
    img = cv2.imread(IMAGE_PATH)
    assert img is not None, f"Không đọc được ảnh: {IMAGE_PATH}"
    h_orig, w_orig = img.shape[:2]
    t_start = time.perf_counter()

    # ── Step 2: YOLO detect ───────────────────────────────────────────
    t0            = time.perf_counter()
    yolo_res      = yolo.predict(img, conf=CONF_THRESHOLD, verbose=False)
    t_yolo        = (time.perf_counter() - t0) * 1000
    boxes         = yolo_res[0].boxes
    box_xyxy      = None
    yolo_conf     = 0.0
    yolo_detected = False
    crop          = None

    if boxes is not None and len(boxes) > 0:
        xyxy  = boxes.xyxy.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        areas = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
        best  = np.argmax(areas)
        x1, y1, x2, y2 = xyxy[best].astype(int)
        x1 = max(0, x1 - PADDING);      y1 = max(0, y1 - PADDING)
        x2 = min(w_orig, x2 + PADDING); y2 = min(h_orig, y2 + PADDING)
        yolo_conf = float(confs[best])

        if yolo_conf < 0.6:
            print(f" YOLO detect không rõ (conf={yolo_conf*100:.0f}% < 60%) — không thể dự đoán")
        else:
            box_xyxy      = (x1, y1, x2, y2)
            yolo_detected = True
            crop          = img[y1:y2, x1:x2]
    else:
        print(" YOLO không tìm thấy lá lúa trong ảnh — không thể dự đoán")

    # ── Step 3 → 4: Chỉ chạy nếu YOLO detect thành công ─────────────
    if yolo_detected and crop is not None:

        # Resize crop → 224×224
        crop_224 = cv2.resize(crop, (224, 224))

        # MobileNet classify
        t1      = time.perf_counter()
        img_rgb = cv2.cvtColor(crop_224, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        probs   = mobilenet.predict(np.expand_dims(img_rgb, 0), verbose=0)[0]
        t_cls   = (time.perf_counter() - t1) * 1000
        t_total = (time.perf_counter() - t_start) * 1000

        pred_idx   = int(np.argmax(probs))
        pred_class = CLASS_NAMES[pred_idx]
        pred_conf  = float(probs[pred_idx])
        info       = CLASS_INFO[pred_class]

        if pred_conf < 0.70:
            print(f" Độ tin cậy thấp ({pred_conf*100:.1f}% < 70%) — không thể xác định bệnh")
        else:
            # ── In kết quả ────────────────────────────────────────────
            print("\n" + "═"*55)
            print("  KẾT QUẢ DỰ ĐOÁN")
            print("═"*55)
            print(f"  Bệnh (EN)        : {pred_class}")
            print(f"  Bệnh (VI)        : {info['vi']}")
            print(f"  Độ tin cậy       : {pred_conf*100:.1f}%")
            print(f"  Mức độ nguy hiểm : {info['severity']}")
            print(f"  YOLO detect      : Có ({yolo_conf*100:.0f}%)")
            print(f"  Latency YOLO     : {t_yolo:.0f} ms")
            print(f"  Latency MNet     : {t_cls:.0f} ms")
            print(f"  Tổng latency     : {t_total:.0f} ms")
            print("═"*55)

            # ── Visualize ─────────────────────────────────────────────
            fig = plt.figure(figsize=(16, 5))

            # [A] Ảnh gốc + bbox
            ax1 = fig.add_subplot(1, 3, 1)
            ax1.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            x1, y1, x2, y2 = box_xyxy
            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=3, edgecolor='#00ff88', facecolor='none'
            )
            ax1.add_patch(rect)
            ax1.text(
                x1, max(y1-8, 0),
                f"rice_leaf  {yolo_conf*100:.0f}%",
                color='white', fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', fc='#00aa55', ec='none', alpha=0.85)
            )
            ax1.set_title(f"① Ảnh gốc\nYOLO: detected ({yolo_conf*100:.0f}%)", fontsize=10)
            ax1.axis('off')

            # [B] Crop 224×224
            ax2 = fig.add_subplot(1, 3, 2)
            ax2.imshow(cv2.cvtColor(crop_224, cv2.COLOR_BGR2RGB))
            ax2.set_title('② Crop 224×224\n→ input MobileNet', fontsize=10)
            ax2.axis('off')

            # [C] Top-5 probability bar
            ax3      = fig.add_subplot(1, 3, 3)
            top5_idx   = np.argsort(probs)[::-1][:5]
            top5_probs = probs[top5_idx]
            top5_names = [CLASS_NAMES[i] for i in top5_idx]
            bar_colors = [info['color'] if n == pred_class else '#95a5a6' for n in top5_names]
            bars = ax3.barh(top5_names[::-1], top5_probs[::-1] * 100, color=bar_colors[::-1])
            ax3.set_xlabel('Probability (%)', fontsize=9)
            ax3.set_title('③ Top-5 Predictions', fontsize=10)
            ax3.set_xlim(0, 105)
            for bar, prob in zip(bars, top5_probs[::-1]):
                ax3.text(
                    prob*100 + 0.5, bar.get_y() + bar.get_height()/2,
                    f'{prob*100:.1f}%', va='center', fontsize=8
                )

            plt.suptitle(
                f"{pred_class}  ({pred_conf*100:.1f}%)\n{info['vi']}  —  Mức độ: {info['severity']}",
                fontsize=13, fontweight='bold', color=info['color']
            )
            plt.tight_layout()
            plt.savefig('prediction_result.png', dpi=120, bbox_inches='tight')
            plt.show()
            print(" Đã lưu → prediction_result.png")